# 08 - Visualizaciones

Análisis visual del dataset y resultados del modelo.
Desarrollado con Plotly para interactividad.

## 1. Setup y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import requests
from sklearn.metrics import precision_recall_curve, average_precision_score
from plotly.subplots import make_subplots
from supabase import create_client
from dotenv import load_dotenv
import joblib
import os

load_dotenv(dotenv_path='../.env')

supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_KEY")
)

todos = []
page = 0
while True:
    response = supabase.table('asteroids')\
        .select('*')\
        .range(page * 1000, (page + 1) * 1000 - 1)\
        .execute()
    if not response.data:
        break
    todos.extend(response.data)
    page += 1

df = pd.DataFrame(todos)
print(f"Datos cargados: {df.shape[0]} filas")

Datos cargados: 22627 filas


## 2. Qué vemos en los datos

### Velocidad vs distancia de acercamiento

In [2]:
fig = px.scatter(
    df,
    x='miss_distance_km',
    y='velocity_km_per_hour',
    color='is_potentially_hazardous',
    color_discrete_map={True: '#FF4136', False: '#0074D9'},
    labels={
        'miss_distance_km': 'Distancia de acercamiento (km)',
        'velocity_km_per_hour': 'Velocidad (km/h)',
        'is_potentially_hazardous': 'Peligroso'
    },
    title='Velocidad vs distancia de acercamiento',
    opacity=0.5,
    hover_data=['name', 'absolute_magnitude_h']
)

fig.update_layout(height=600)
fig.show()

### Findings

Los asteroides peligrosos no forman un cluster separado en el espacio
velocidad/distancia. Se distribuyen a lo largo de todo el rango de
ambas variables, confirmando lo observado en el EDA: miss_distance_km
y velocity_km_per_hour tienen bajo poder discriminativo por sí solas.

Una concentración leve de peligrosos aparece en velocidades superiores
a 75.000 km/h, consistente con el mayor SHAP value positivo de velocity
observado en el análisis de interpretabilidad. Sin embargo el solapamiento
es significativo y ninguna combinación de estas dos variables sería
suficiente para clasificar correctamente.

Esto refuerza la conclusión del modelo: absolute_magnitude_h (tamaño)
es la feature con mayor poder discriminativo real.

### Timeline de asteroides detectados por mes

In [3]:
df['close_approach_date'] = pd.to_datetime(df['close_approach_date'])
df['year_month'] = df['close_approach_date'].dt.to_period('M').astype(str)

timeline = df.groupby(['year_month', 'is_potentially_hazardous']).size().reset_index(name='count')

fig = px.bar(
    timeline,
    x='year_month',
    y='count',
    color='is_potentially_hazardous',
    color_discrete_map={True: '#FF4136', False: '#0074D9'},
    labels={
        'year_month': 'Mes',
        'count': 'Asteroides detectados',
        'is_potentially_hazardous': 'Peligroso'
    },
    title='Asteroides detectados por mes (2021-2026)'
)

fig.update_layout(height=500, xaxis_tickangle=45)
fig.show()

### Findings

La distribución temporal no es uniforme. 2021 presenta el mayor volumen
de detecciones del período analizado, con picos de hasta 600 asteroides
por mes versus 200-300 en años posteriores.

Al notar esta anomalía, investigué si había una causa documentada.
En agosto de 2021 la NASA alcanzó el hito del asteroide número 1.000
observado por radar planetario desde 1968, reflejo de una capacidad
de detección en expansión continua.

El programa NEO de NASA está activamente enfocado en identificar el
90% de los objetos mayores a 140 metros, lo que genera picos de
actividad de detección en ciertos períodos. El volumen decreciente
hacia 2026 responde parcialmente a que los datos más recientes no
estaban completamente procesados al momento de la extracción.

Este tipo de análisis contextual es parte del proceso. Un dato que
parece un error puede ser un hallazgo real sobre cómo se generan
los datos.

## 3. Qué aprendió el modelo

### Magnitud absoluta vs probabilidad de peligrosidad

In [15]:
checkpoint = joblib.load('../data/xgboost_final.pkl')
modelo = checkpoint['modelo']

X_test = pd.read_csv('../data/X_test.csv')
y_test = pd.read_csv('../data/y_test.csv').squeeze()

probabilidades = modelo.predict_proba(X_test)[:, 1]

df_test_original = df[df.index.isin(X_test.index)].copy()
df_test_original['probabilidad_peligroso'] = probabilidades
df_test_original['peligroso_real'] = y_test.values
y_pred = (probabilidades >= 0.20).astype(bool)

def clasificar(row):
    if row['peligroso_real'] and row['probabilidad_peligroso'] >= 0.20:
        return 'Verdadero positivo'
    elif row['peligroso_real'] and row['probabilidad_peligroso'] < 0.20:
        return 'Falso negativo'
    elif not row['peligroso_real'] and row['probabilidad_peligroso'] >= 0.20:
        return 'Falso positivo'
    else:
        return 'Verdadero negativo'

df_test_original['clasificacion'] = df_test_original.apply(clasificar, axis=1)

fig = px.scatter(
    df_test_original,
    x='absolute_magnitude_h',
    y='probabilidad_peligroso',
    color='clasificacion',
    color_discrete_map={
        'Verdadero positivo': '#FF4136',
        'Falso negativo': '#FF851B',
        'Falso positivo': '#0074D9',
        'Verdadero negativo': '#AAAAAA'
    },
    labels={
        'absolute_magnitude_h': 'Magnitud absoluta (menor = más grande)',
        'probabilidad_peligroso': 'Probabilidad asignada por el modelo',
        'clasificacion': 'Clasificación'
    },
    title='Magnitud absoluta vs probabilidad de peligrosidad',
    opacity=0.6,
    hover_data=['name', 'diameter_max_km']
)

fig.add_hline(y=0.20, line_dash='dash', line_color='black',
              annotation_text='Umbral 0.20')

fig.update_layout(height=600)
fig.show()

### Findings

La visualización confirma la relación aprendida por el modelo: a medida
que la magnitud absoluta baja (asteroides más grandes), la probabilidad
asignada tiende a subir. Esto es consistente con los SHAP values del
notebook 06 donde absolute_magnitude_h fue la feature dominante.

Los falsos negativos (naranja) aparecen por debajo del umbral 0.20
con magnitudes relativamente bajas. Son los casos estructuralmente
difíciles: asteroides grandes cuyas features de velocidad y distancia
generaron señales contradictorias que el modelo no pudo resolver.

Los falsos positivos (azul) se concentran en el rango de magnitudes
altas (asteroides pequeños). El modelo interpreta combinaciones de
velocidad alta o distancia baja como señal de peligro aunque el objeto
sea pequeño. El trade-off es asimétrico y aceptable: investigar
asteroides pequeños innecesariamente tiene menor costo que ignorar
uno grande realmente peligroso.

### Top 15 asteroides con mayor probabilidad de ser peligrosos

In [4]:
X_full = pd.read_csv('../data/X_test.csv')
y_full = pd.read_csv('../data/y_test.csv').squeeze()

checkpoint = joblib.load('../data/xgboost_final.pkl')
modelo = checkpoint['modelo']

probabilidades = modelo.predict_proba(X_full)[:, 1]

df_test_original = df[df.index.isin(X_full.index)].copy()
df_test_original['probabilidad_peligroso'] = probabilidades
df_test_original['peligroso_real'] = y_full.values

top15 = df_test_original.nlargest(15, 'probabilidad_peligroso')[[
    'name', 'absolute_magnitude_h', 'diameter_max_km',
    'velocity_km_per_hour', 'miss_distance_km',
    'probabilidad_peligroso', 'peligroso_real'
]]

fig = px.bar(
    top15,
    x='probabilidad_peligroso',
    y='name',
    orientation='h',
    color='peligroso_real',
    color_discrete_map={True: '#FF4136', False: '#0074D9'},
    labels={
        'probabilidad_peligroso': 'Probabilidad de ser peligroso',
        'name': 'Asteroide',
        'peligroso_real': 'Peligroso real'
    },
    title='Top 15 asteroides con mayor probabilidad de ser peligrosos',
    hover_data=['absolute_magnitude_h', 'diameter_max_km', 
                'velocity_km_per_hour', 'miss_distance_km']
)

fig.update_layout(height=600, yaxis={'categoryorder': 'total ascending'})
fig.show()

### Findings

Los 15 asteroides con mayor probabilidad asignada por el modelo
son todos verdaderos positivos: el campo is_potentially_hazardous
de la API de NASA confirma la clasificación en todos los casos.

Ninguno de estos objetos aparece en cobertura mediática, a diferencia
de asteroides ampliamente monitoreados como Apophis o Bennu. Esto
refleja una característica real del espacio de datos: la mayoría de
los asteroides técnicamente peligrosos según los criterios de NASA/JPL
(diámetro mayor a 140m, distancia orbital menor a 0.05 AU) son objetos
pequeños y relativamente recientes que no generan alertas públicas
pero sí están catalogados en el CNEOS.

El modelo está aprendiendo los criterios técnicos de clasificación
de NASA, no la notoriedad mediática del objeto.
Fuente: [NASA/JPL CNEOS](https://cneos.jpl.nasa.gov/ca/)

## 4. Qué también aprendió el modelo

### Curva Precision - Recall

In [ ]:
precision_curve, recall_curve, thresholds = precision_recall_curve(y_test, probabilidades)
auc_pr = average_precision_score(y_test, probabilidades)

df_curve = pd.DataFrame({
    'precision': precision_curve[:-1],
    'recall': recall_curve[:-1],
    'umbral': thresholds
})

fig = px.line(
    df_curve,
    x='recall',
    y='precision',
    title=f'Curva Precision-Recall (AUC-PR = {auc_pr:.3f})',
    labels={
        'recall': 'Recall',
        'precision': 'Precision'
    },
    hover_data=['umbral']
)

# Marcamos el umbral elegido
umbral_elegido = df_curve.iloc[(df_curve['umbral'] - 0.20).abs().argsort()[:1]]
fig.add_scatter(
    x=umbral_elegido['recall'],
    y=umbral_elegido['precision'],
    mode='markers',
    marker=dict(size=12, color='red', symbol='star'),
    name='Umbral 0.20'
)

fig.update_layout(height=500)
fig.show()

print(f"AUC-PR: {auc_pr:.3f}")
p = umbral_elegido['precision'].values[0]
r = umbral_elegido['recall'].values[0]
print(f"En umbral 0.20: Precision={p:.3f}, Recall={r:.3f}")

AUC-PR: 0.588
En umbral 0.20: Precision=0.369, Recall=0.901


### Findings

AUC-PR de 0.588 sobre una baseline aleatoria de 0.056 para esta
proporción de clases. El modelo tiene poder discriminativo real
aunque limitado por las 4 features disponibles.

La curva muestra una caída suave hasta recall 0.80 donde la precision
se mantiene por encima de 0.40. El umbral 0.20 (estrella roja) fue
seleccionado en la zona de recall 0.90, priorizando la detección
completa sobre la precision. En un contexto de defensa planetaria
este es el trade-off correcto.

Para mejorar el AUC-PR se requieren features orbitales adicionales,
no más tuning del modelo actual.

### Validación externa: cruce con datos oficiales NASA/JPL

Cruce del top 100 del modelo contra la lista oficial de close approaches
de NASA/JPL CNEOS para verificar si el modelo identifica los mismos
objetos que NASA considera técnicamente peligrosos.

Fuente: [NASA/JPL Small Body Database API](https://ssd-api.jpl.nasa.gov/doc/cad.html)

In [ ]:
# Top 100 del modelo
top100 = df_test_original.nlargest(100, 'probabilidad_peligroso')[
    ['name', 'absolute_magnitude_h', 'diameter_max_km',
     'probabilidad_peligroso', 'peligroso_real']
].copy()

top100['name_clean'] = top100['name'].str.replace('(', '').str.replace(')', '').str.strip()

def extraer_designacion(nombre):
    partes = nombre.strip().split()
    if len(partes) >= 3 and partes[0].isdigit() and len(partes[0]) > 4:
        return ' '.join(partes[1:])
    return nombre.strip()

top100['designacion'] = top100['name_clean'].apply(extraer_designacion)

# Consultamos API de JPL
url = "https://ssd-api.jpl.nasa.gov/cad.api"
params = {
    'dist-max': '0.05',
    'h-max': '22',
    'date-min': '2021-01-01',
    'date-max': '2026-04-01',
    'sort': 'h'
}

response = requests.get(url, params=params)
data = response.json()
df_jpl = pd.DataFrame(data['data'], columns=data['fields'])

# Cruce
jpl_des = set(df_jpl['des'].str.strip().tolist())
modelo_des = set(top100['designacion'].tolist())
en_ambos = modelo_des.intersection(jpl_des)

print(f"Top 100 del modelo: {len(top100)}")
print(f"Lista JPL: {len(jpl_des)} designaciones únicas")
print(f"En ambas listas: {len(en_ambos)}")

Top 100 del modelo: 100
Lista JPL: 170 designaciones únicas
En ambas listas: 0


### Findings

El cruce entre el top 100 del modelo y la base de datos de close approaches
de NASA/JPL CNEOS dio 0 coincidencias.

La causa es que NeoWs y CNEOS son sistemas distintos con criterios
de indexación diferentes. NeoWs clasifica is_potentially_hazardous
basándose en parámetros orbitales históricos calculados por JPL.
CNEOS close approach API lista objetos con acercamientos activos
dentro de ventanas de tiempo específicas, usando IDs numéricos
permanentes en lugar de designaciones temporales.

Esto no invalida las predicciones del modelo. El campo
is_potentially_hazardous que usamos como target proviene
directamente de la clasificación oficial de JPL embebida en NeoWs.
La fuente de verdad es la misma, los sistemas de consulta son distintos.

Para verificar individualmente cada asteroide del top 100, la fuente
correcta es el [JPL Small Body Database](https://ssd.jpl.nasa.gov/tools/sbdb_lookup.html)
buscando por designación.